In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from gensim.models import Word2Vec
import gensim
from nltk.tokenize import sent_tokenize, word_tokenize
import warnings
import gensim.downloader
import os, glob
from collections import defaultdict
from tqdm import tqdm
import pandas as pd
import numpy as np
from collections import Counter
import sys
import math
import torch.optim
from skl2onnx import to_onnx
from nltk.corpus import words
import yaml
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.datasets import make_classification

torch.manual_seed(1)

In [2]:
def get_glove_vectors():
    glove_vectors = gensim.downloader.load('glove-wiki-gigaword-300')
    words_ = words.words()
    common_words = set([x for x in words_ if glove_vectors.has_index_for(x)])
    return glove_vectors, common_words

glove_vectors, common_words = get_glove_vectors()

In [3]:
def remove_bracket(string):
    if '[' not in string:
        return string, 0
    return string[:string.index('[')] + string[string.index(']')+1:], eval(string[string.index('[')+1:string.index(']')])

def get_text_embedding(stack, layers=4, word_limit=20):
    stack = [x.split(' ')[:word_limit] for x in stack.split('\n')][:layers]
    lengths = torch.Tensor([x.index('*') if '*' in x else len(x) for x in stack])
    embedding = torch.Tensor(np.array([[glove_vectors.get_vector(x) for x in embedding_i] for embedding_i in stack]))
        
    # https://stackoverflow.com/questions/53403306/how-to-batch-convert-sentence-lengths-to-masks-in-pytorch
    mask = torch.arange(word_limit).expand(len(lengths), word_limit) < lengths.unsqueeze(1)
    return embedding, mask

In [4]:
df = defaultdict(list)
labels_to_index = defaultdict(lambda: -1)
labels_set = []
dataset_regex = "/Users/ginoprasad/Job_Applications/web_crawler/dataset/*/keystrokes/page*.txt"
for path in tqdm(glob.glob(dataset_regex)):
    with open(path) as infile:
        infile_lines = infile.readlines()
        
        x_values = [x.strip().split('\n') for x in ''.join(infile_lines).split(('-'*50) + '\n')[1::2]]
        raw_y_values = [x.strip() for x in infile_lines[18::20]]
        assert len(x_values) == len(raw_y_values)
        for x_value, raw_y_value in zip(x_values, raw_y_values):
            y_value, index = remove_bracket(raw_y_value)
            if y_value not in labels_to_index:
                labels_to_index[y_value] = len(labels_to_index)
                labels_set.append(y_value)
            df['path'].append(path)
            df['lines'].append('\n'.join(x_value))
            df['label_value'].append(y_value)
            df['label'].append(labels_to_index[y_value])
            df['index'].append(index)
            df['raw_label'].append(raw_y_value)
df = pd.DataFrame(df)
df

100%|█████| 16/16 [00:00<00:00, 1724.32it/s]


,path,lines,label_value,label,index,raw_label
0,/Users/ginoprasad/Job_Applications/web_crawler...,^ name * * * * * * * * * * * * * * * * * * * *...,FULL_NAME,0,0,FULL_NAME
1,/Users/ginoprasad/Job_Applications/web_crawler...,^ m m * * * * * * * * * * * * * * * * * * * * ...,TODAY.MONTH,1,0,TODAY.MONTH
2,/Users/ginoprasad/Job_Applications/web_crawler...,^ d d * * * * * * * * * * * * * * * * * * * * ...,TODAY.DAY,2,0,TODAY.DAY
3,/Users/ginoprasad/Job_Applications/web_crawler...,^ y y y y * * * * * * * * * * * * * * * * * * ...,TODAY.YEAR,3,0,TODAY.YEAR
4,/Users/ginoprasad/Job_Applications/web_crawler...,^ no i do not have a disability and have not h...,YES,4,0,YES
...,...,...,...,...,...,...
141,/Users/ginoprasad/Job_Applications/web_crawler...,^ make a selection yes no * * * * * * * * * * ...,NO,20,0,NO
142,/Users/ginoprasad/Job_Applications/web_crawler...,^ make a selection male female opt out * * * *...,GENDER,18,0,GENDER
143,/Users/ginoprasad/Job_Applications/web_crawler...,^ make a selection hispanic or latino american...,DEMOGRAPHIC,19,0,DEMOGRAPHIC
144,/Users/ginoprasad/Job_Applications/web_crawler...,^ if you believe you belong to any of the cate...,NO_VETERAN,17,0,NO_VETERAN


In [31]:
profile_path = '/Users/ginoprasad/Job_Applications/Profiles/default.yaml'
with open(profile_path) as infile:
    profile = yaml.safe_load(infile)

In [40]:
labels_path = "/Users/ginoprasad/Job_Applications/web_crawler/genie/models/labels.txt"
with open(labels_path, 'w') as outfile:
    for label in labels_set:
#         max_index = max(df[df['label_value'] == label]['index'])
        length = int(type(profile[label.split('.')[0]]) != list) or len(profile[label.split('.')[0]])
        outfile.write(f"{label},{length}\n")

In [21]:
train_x_src, train_x_mask, train_y = [], [], []
for elem in tqdm(df.iloc):
    src, mask = get_text_embedding(elem['lines'])
    train_x_src.append(src)
    train_x_mask.append(~mask)
    train_y.append(elem['label'])
train_x_src = torch.Tensor(torch.stack(train_x_src))
train_x_mask = torch.Tensor(torch.stack(train_x_mask))
train_y = torch.Tensor(train_y).type(torch.int64)

146it [00:00, 3724.69it/s]


### Training

In [297]:
X = train_x_src.reshape((train_x_src.size(0), -1)).numpy()
y = train_y.numpy()

In [298]:
model = HistGradientBoostingClassifier(random_state=0, verbose=1)

In [299]:
model.fit(X, y)

Binning 0.028 GB of training data: 0.906 s
Fitting gradient boosted rounds:
[1/100] 37 trees, 76 leaves (2 on avg), max depth = 3, in 1.638s
[2/100] 37 trees, 225 leaves (6 on avg), max depth = 5, in 4.160s
[3/100] 37 trees, 225 leaves (6 on avg), max depth = 6, in 4.247s
[4/100] 37 trees, 222 leaves (6 on avg), max depth = 6, in 4.048s
[5/100] 37 trees, 227 leaves (6 on avg), max depth = 6, in 4.018s
[6/100] 37 trees, 223 leaves (6 on avg), max depth = 6, in 4.111s
[7/100] 37 trees, 218 leaves (5 on avg), max depth = 6, in 4.093s
[8/100] 37 trees, 223 leaves (6 on avg), max depth = 6, in 4.208s
[9/100] 37 trees, 220 leaves (5 on avg), max depth = 6, in 4.091s
[10/100] 37 trees, 218 leaves (5 on avg), max depth = 5, in 4.859s
[11/100] 37 trees, 221 leaves (5 on avg), max depth = 6, in 4.080s
[12/100] 37 trees, 222 leaves (6 on avg), max depth = 5, in 4.077s
[13/100] 37 trees, 219 leaves (5 on avg), max depth = 5, in 4.626s
[14/100] 37 trees, 219 leaves (5 on avg), max depth = 6, in 4.8

HistGradientBoostingClassifier(random_state=0, verbose=1)

In [314]:
onx = to_onnx(model, X[:1].astype(np.float32), target_opset=12)
with open("models/model.onnx", "wb") as f:
    f.write(onx.SerializeToString())

## Evaluation

In [143]:
import genie_v1
import importlib
genie_v1 = importlib.reload(genie_v1)
genie_v1.GLOBALS['glove_vectors'] = glove_vectors

In [144]:
genie_v1.test()

'Developed Image Processing and Computer Vision Tools for smFISH (Fluorescence in Situ Hybridization) Image Data. Technical Skills: Python with Tensorflow, OpenCV, Numpy, Pandas, Linux, Git. Web development for AmpliconRepository, an ecDNA (extra‐chromosomalDNA) Public Web Database. Technical Skills: MongoDB and Django for web framework, querying functionality using Python’s SQLite3.'